# LAB 4 — Git Basics for a Python Project**Student:** Prabhas P Simha  **USN:** 24SSDS415036  **GitHub:** https://github.com/Pubby-15  **Repository:** https://github.com/Pubby-15/ADV-Lab-05Run the cells in order in Google Colab or Jupyter. This notebook demonstrates Git identity, repository inspection, staging, commits, remote push, feature branching, and Streamlit deployment preparation. Never store a Personal Access Token in a notebook or source file.

## Task 1 — Connect to the existing GitHub repository

In [ ]:
import osimport subprocessfrom pathlib import PathGITHUB_USER = "Pubby-15"REPO_NAME = "ADV-Lab-05"REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"if not Path(".git").exists():    if not Path(REPO_NAME).exists():        subprocess.run(["git", "clone", REPO_URL, REPO_NAME], check=True)    os.chdir(REPO_NAME)def git(*args, check=True):    return subprocess.run(["git", *args], check=check, text=True)print("Working directory:", Path.cwd())git("status")

## Task 2 — Configure Git identity

In [ ]:
student_name = "Prabhas P Simha"github_email = input("Enter your GitHub-associated email: ").strip()if not github_email:    raise ValueError("A GitHub email is required.")git("config", "--global", "user.name", student_name)git("config", "--global", "user.email", github_email)git("config", "--global", "--list")

## Task 3 — Inspect project structure and ignore rules

In [ ]:
print("\n".join(sorted(p.name for p in Path(".").iterdir())))print("\n.gitignore contents:")print(Path(".gitignore").read_text(encoding="utf-8"))

## Task 4 — Stage, commit, and inspect the project history

In [ ]:
git("add", "app.py", "requirements.txt", "README.md", ".gitignore")git("status")# Commit only when there are staged changes.staged = subprocess.run(["git", "diff", "--cached", "--quiet"], check=False)if staged.returncode != 0:    git("commit", "-m", "Update dashboard project")else:    print("No staged changes to commit.")git("log", "--oneline", "-5")

## Task 5 — Verify remote and push changes securelyThe remote is already configured for this repository. On a Colab session that does not already have GitHub credentials, authenticate using a token entered through a hidden prompt. Never put the token directly in a command, notebook output, source file, or remote URL.

In [ ]:
import getpassimport tempfileremote_url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"git("remote", "set-url", "origin", remote_url)def push_with_hidden_token(arguments):    token = getpass.getpass("GitHub token (hidden): ")    if not token:        raise ValueError("A token is required for HTTPS authentication.")    askpass = Path(tempfile.gettempdir()) / "git_askpass_lab4.sh"    askpass.write_text(        '#!/bin/sh\n'        'case "$1" in\n'        '  *sername*) printf "%s\\n" "$GITHUB_USERNAME" ;;\n'        '  *assword*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'        '  *) printf "\\n" ;;\n'        'esac\n',        encoding="utf-8",    )    askpass.chmod(0o700)    env = os.environ.copy()    env.update({        "GIT_ASKPASS": str(askpass),        "GIT_ASKPASS_REQUIRE": "force",        "GIT_TERMINAL_PROMPT": "0",        "GITHUB_USERNAME": GITHUB_USER,        "GITHUB_TOKEN": token,    })    try:        subprocess.run(["git", *arguments], env=env, check=True)    finally:        askpass.unlink(missing_ok=True)        env.pop("GITHUB_TOKEN", None)        token = Nonepush_with_hidden_token(["push", "origin", "main"])

## Task 6 — Everyday Git workflow

In [ ]:
# Use this sequence after editing an application file.git("status")# git("add", "app.py")# git("commit", "-m", "Update dashboard")# git("push", "origin", "main")git("log", "--oneline", "-5")

## Task 7 — Create and publish a feature branch

In [ ]:
branch_name = "feature/experiment"branches = subprocess.run(    ["git", "branch", "--list", branch_name],    capture_output=True,    text=True,    check=True,).stdout.strip()if branches:    git("switch", branch_name)else:    git("switch", "-c", branch_name)feature_file = Path("feature_demo.py")feature_file.write_text(    '"""Small feature-branch example for the Git Basics lab."""\n\n'    'def make_feature_summary(year: int, continents: list[str]) -> str:\n'    '    regions = ", ".join(continents) if continents else "no continents selected"\n'    '    return f"Feature preview for {year}: {regions}"\n\n'    'if __name__ == "__main__":\n'    '    print(make_feature_summary(2007, ["Asia", "Europe"]))\n',    encoding="utf-8",)git("add", "feature_demo.py")pending = subprocess.run(["git", "diff", "--cached", "--quiet"], check=False)if pending.returncode != 0:    git("commit", "-m", "Add feature branch demonstration")push_with_hidden_token(["push", "-u", "origin", branch_name])print("Open a pull request from feature/experiment into main on GitHub.")

## Task 8 — Prepare the Streamlit dashboard

In [ ]:
git("switch", "main")print(Path("requirements.txt").read_text(encoding="utf-8"))print("Install: pip install -r requirements.txt")print("Run: streamlit run app.py")

### Streamlit Community Cloud deployment checklist1. Open https://share.streamlit.io/ and sign in with GitHub.2. Create a new app and select `Pubby-15/ADV-Lab-05`.3. Select the `main` branch and set `app.py` as the main file.4. Deploy and check that the dependencies install successfully.5. Save the generated `.streamlit.app` URL after deployment succeeds.**Status:** The repository source and dependencies are prepared. A live app URL remains pending until the deployment is completed in Streamlit Community Cloud.## ConclusionThis notebook demonstrates Git identity configuration, staging, commits, remote synchronization, feature branching, and application deployment preparation. Use meaningful commit messages and keep credentials out of version-controlled files.